# RAG Pipeline Evaluation — Visualizations

This notebook reads the final evaluation results and generates charts for the thesis.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import numpy as np

matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 150})
sns.set_theme(style="whitegrid")

RESULTS_DIR = PROJECT_ROOT / "data" / "results" / "final"
FIGURES_DIR = PROJECT_ROOT / "data" / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

summary_path = RESULTS_DIR / "deterministic_eval_summary.csv"
details_path = RESULTS_DIR / "deterministic_eval_results.csv"

summary = pd.read_csv(summary_path)
details = pd.read_csv(details_path)

IMPL_ORDER = ["basic", "vectordb", "vectordb_tuned", "reranker", "hybrid", "hybrid_rerank"]
IMPL_LABELS = {
    "basic": "Basic",
    "vectordb": "VectorDB",
    "vectordb_tuned": "VectorDB (Tuned)",
    "reranker": "Reranker",
    "hybrid": "Hybrid",
    "hybrid_rerank": "Hybrid + Rerank",
}

available = [i for i in IMPL_ORDER if i in summary["implementation"].values]
summary = summary.set_index("implementation").loc[available].reset_index()
summary["label"] = summary["implementation"].map(IMPL_LABELS)

print(f"Loaded {len(summary)} implementations, {len(details)} detail rows")
summary

## 1. Answer Quality Comparison

In [ ]:
quality_metrics = ["manual_score", "rougeL_f1", "bertscore_f1", "semantic_similarity"]
quality_labels = ["Manual Score", "ROUGE-L F1", "BERTScore F1", "Semantic Similarity"]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(available))
width = 0.18

for i, (metric, label) in enumerate(zip(quality_metrics, quality_labels)):
    if metric in summary.columns:
        vals = summary[metric].values
        bars = ax.bar(x + i * width, vals, width, label=label)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xlabel("Pipeline")
ax.set_ylabel("Score")
ax.set_title("Answer Quality Comparison Across RAG Pipelines")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(summary["label"], rotation=15, ha="right")
ax.set_ylim(0, 1.15)
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "answer_quality.png", bbox_inches="tight")
plt.show()

## 2. Retrieval Quality Comparison

In [ ]:
retrieval_metrics = ["retrieval_hit", "retrieval_precision", "retrieval_recall"]
retrieval_labels = ["Hit Rate", "Precision", "Recall"]

if "retrieval_f1" in summary.columns:
    retrieval_metrics.append("retrieval_f1")
    retrieval_labels.append("F1")

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(available))
width = 0.18

for i, (metric, label) in enumerate(zip(retrieval_metrics, retrieval_labels)):
    if metric in summary.columns:
        vals = summary[metric].values
        bars = ax.bar(x + i * width, vals, width, label=label)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xlabel("Pipeline")
ax.set_ylabel("Score")
ax.set_title("Retrieval Quality Comparison Across RAG Pipelines")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(summary["label"], rotation=15, ha="right")
ax.set_ylim(0, 1.15)
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "retrieval_quality.png", bbox_inches="tight")
plt.show()

## 3. Scenario-Level Heatmap

In [ ]:
details["scenario_num"] = details["scenario_id"].str.extract(r"(\d+)").astype(int)

scenario_labels = {
    1: "S1: Fact Retrieval",
    2: "S2: Procedural",
    3: "S3: Multi-team",
    4: "S4: Policy + Exception",
    5: "S5: Cross-functional",
}

pivot = details.pivot_table(
    index="implementation",
    columns="scenario_num",
    values="manual_score",
    aggfunc="mean",
)

pivot = pivot.reindex([i for i in IMPL_ORDER if i in pivot.index])
pivot = pivot.rename(index=IMPL_LABELS, columns=scenario_labels)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Manual Score by Pipeline and Scenario")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "scenario_heatmap.png", bbox_inches="tight")
plt.show()

## 4. Timing Breakdown

In [ ]:
time_cols = ["indexing_time_s", "retrieval_time_s", "generation_time_s"]
time_labels = ["Indexing", "Retrieval", "Generation"]

has_timing = all(c in summary.columns for c in time_cols)

if has_timing:
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(available))
    bottom = np.zeros(len(available))

    colors = ["#4e79a7", "#f28e2b", "#e15759"]

    for col, label, color in zip(time_cols, time_labels, colors):
        vals = summary[col].fillna(0).values
        ax.bar(x, vals, 0.5, bottom=bottom, label=label, color=color)
        for i, val in enumerate(vals):
            if val > 0:
                ax.text(x[i], bottom[i] + val / 2, f"{val:.1f}s",
                        ha="center", va="center", fontsize=8, color="white", fontweight="bold")
        bottom += vals

    ax.set_xlabel("Pipeline")
    ax.set_ylabel("Time (seconds)")
    ax.set_title("Average Timing Breakdown per Query")
    ax.set_xticks(x)
    ax.set_xticklabels(summary["label"], rotation=15, ha="right")
    ax.legend(loc="upper right")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "timing_breakdown.png", bbox_inches="tight")
    plt.show()
else:
    print("Timing columns not found in summary. Re-run deterministic_eval.py first.")

## 5. Radar Chart — Multi-Metric Profile

In [ ]:
radar_metrics = ["manual_score", "bertscore_f1", "semantic_similarity", "retrieval_recall", "retrieval_precision"]
radar_labels = ["Manual Score", "BERTScore F1", "Semantic Sim.", "Retrieval Recall", "Retrieval Precision"]

radar_metrics = [m for m in radar_metrics if m in summary.columns]
radar_labels = radar_labels[:len(radar_metrics)]

angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = plt.cm.Set2(np.linspace(0, 1, len(available)))

for idx, impl in enumerate(available):
    row = summary[summary["implementation"] == impl]
    values = [row[m].values[0] for m in radar_metrics]
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2, label=IMPL_LABELS.get(impl, impl), color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_thetagrids(np.degrees(angles[:-1]), radar_labels)
ax.set_ylim(0, 1)
ax.set_title("Multi-Metric Profile per Pipeline", y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "radar_chart.png", bbox_inches="tight")
plt.show()

## Summary

All figures saved to `data/results/figures/`.